### Learning Objectives
- Master all DataFrame transformations
- Master all DataFrame actions
- Understand lazy evaluation deeply
- Identify which operations trigger shuffles

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Module03_Transformations_vs_Actions") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.repl.eagerEval.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Load data
orders_df = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
customers_df = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products_df = spark.read.csv("data/products.csv", header=True, inferSchema=True)

print(f"Orders: {orders_df.count()} rows")
print(f"Customers: {customers_df.count()} rows")

Orders: 50000 rows
Customers: 5000 rows


In [3]:
# ============================================================
# 1.1 Narrow Transformations (No Shuffle)
# ============================================================

# map - Apply function to each element
# filter - Filter rows based on condition
# flatMap - Map + flatten
# union - Combine DataFrames with same schema
# mapPartitions - Apply function to each partition

# These are ALL narrow transformations

result = (
    orders_df
    .filter(col("status")=="delivered")
    .select("order_id","customer_id","total_amount")
    .withColumn("amount_category",
                when(col("total_amount")>1000,"High")
                .when(col("total_amount")>500,"Medium").otherwise("Low"))
)

print("This is a lazy transformation - no computation yet!")
print(f"Plan: {result.explain()}...")

This is a lazy transformation - no computation yet!
== Physical Plan ==
*(1) Project [order_id#17, customer_id#18, total_amount#22, CASE WHEN (total_amount#22 > 1000.0) THEN High WHEN (total_amount#22 > 500.0) THEN Medium ELSE Low END AS amount_category#113]
+- *(1) Filter (isnotnull(status#21) AND (status#21 = delivered))
   +- FileScan csv [order_id#17,customer_id#18,status#21,total_amount#22] Batched: false, DataFilters: [isnotnull(status#21), (status#21 = delivered)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/shubh/Desktop/Progress/Developer/PySpark/data/orders.csv], PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,delivered)], ReadSchema: struct<order_id:int,customer_id:int,status:string,total_amount:double>


Plan: None...


In [4]:
# ============================================================
# 1.2 Wide Transformations (Shuffle)
# ============================================================

# groupBy - Group data and aggregate
# join - Combine DataFrames based on key
# distinct - Remove duplicates
# repartition - Change number of partitions
# sortBy / orderBy - Sort data
# cube / rollup - Multi-dimensional aggregation

# groupBy triggers shuffle

grouped_df = orders_df.groupBy("customer_id").agg(
    count("*").alias("order_count"),
    sum("total_amount").alias("total_spent")
)

print("Grouped is a wide transformation (shuffle happens)")
print(f"Number of partition after groupBy: {grouped_df.rdd.getNumPartitions()}")

Grouped is a wide transformation (shuffle happens)
Number of partition after groupBy: 1


In [ ]:
# 1.3 All DataFrame Transformations

# --- SELECTION ---
orders_df.select("order_id","status","total_amount").show(5)

+--------+---------+------------+
|order_id|   status|total_amount|
+--------+---------+------------+
|       1|  Shipped|      1784.2|
|       2|Delivered|     1536.51|
|       3| Returned|     1191.88|
|       4|Delivered|     1157.78|
|       5|Cancelled|      1381.5|
+--------+---------+------------+
only showing top 5 rows


In [7]:
# --- Filter ---
orders_df.filter(col("status")=="Cancelled").show(5)

+--------+-----------+----------+-------------+---------+------------+---------------+----------+-------------+----------------+----------------+
|order_id|customer_id|order_date|delivery_date|   status|total_amount|discount_amount|tax_amount|shipping_cost|  payment_method|shipping_address|
+--------+-----------+----------+-------------+---------+------------+---------------+----------+-------------+----------------+----------------+
|       5|       1330|2023-12-30|   2024-01-03|Cancelled|      1381.5|         213.19|    110.52|        14.99|     Credit Card|     757 Main St|
|       6|       3811|2022-02-24|   2022-02-26|Cancelled|     1117.94|          42.07|     89.44|         9.99|          PayPal|    7348 Main St|
|      13|        977|2022-05-04|         NULL|Cancelled|     2472.23|          191.2|    197.78|        14.99|     Credit Card|    1689 Main St|
|      27|       2868|2022-10-05|         NULL|Cancelled|     1196.81|          14.49|     95.74|         9.99|          Pay

In [8]:
# --- Adding Columns ---
orders_df.withColumn("tax",col("total_amount")*0.08).show(3)

+--------+-----------+----------+-------------+---------+------------+---------------+----------+-------------+----------------+----------------+------------------+
|order_id|customer_id|order_date|delivery_date|   status|total_amount|discount_amount|tax_amount|shipping_cost|  payment_method|shipping_address|               tax|
+--------+-----------+----------+-------------+---------+------------+---------------+----------+-------------+----------------+----------------+------------------+
|       1|       4722|2023-08-08|   2023-08-14|  Shipped|      1784.2|         163.19|    142.74|         5.99|   Bank Transfer|    5823 Main St|142.73600000000002|
|       2|       3172|2022-09-19|   2022-09-24|Delivered|     1536.51|           63.7|    122.92|          0.0|Cash on Delivery|    5796 Main St|          122.9208|
|       3|       1381|2023-03-22|   2023-03-25| Returned|     1191.88|          40.14|     95.35|          0.0|Cash on Delivery|    3354 Main St| 95.35040000000001|
+--------+

In [9]:
# --- Droping Columns ---

orders_df.drop(col("tax")).show(3)

+--------+-----------+----------+-------------+---------+------------+---------------+----------+-------------+----------------+----------------+
|order_id|customer_id|order_date|delivery_date|   status|total_amount|discount_amount|tax_amount|shipping_cost|  payment_method|shipping_address|
+--------+-----------+----------+-------------+---------+------------+---------------+----------+-------------+----------------+----------------+
|       1|       4722|2023-08-08|   2023-08-14|  Shipped|      1784.2|         163.19|    142.74|         5.99|   Bank Transfer|    5823 Main St|
|       2|       3172|2022-09-19|   2022-09-24|Delivered|     1536.51|           63.7|    122.92|          0.0|Cash on Delivery|    5796 Main St|
|       3|       1381|2023-03-22|   2023-03-25| Returned|     1191.88|          40.14|     95.35|          0.0|Cash on Delivery|    3354 Main St|
+--------+-----------+----------+-------------+---------+------------+---------------+----------+-------------+-------------

In [10]:
# --- Rename Column ---

orders_df.withColumnRenamed("total_amount","amount").show(3)

+--------+-----------+----------+-------------+---------+-------+---------------+----------+-------------+----------------+----------------+
|order_id|customer_id|order_date|delivery_date|   status| amount|discount_amount|tax_amount|shipping_cost|  payment_method|shipping_address|
+--------+-----------+----------+-------------+---------+-------+---------------+----------+-------------+----------------+----------------+
|       1|       4722|2023-08-08|   2023-08-14|  Shipped| 1784.2|         163.19|    142.74|         5.99|   Bank Transfer|    5823 Main St|
|       2|       3172|2022-09-19|   2022-09-24|Delivered|1536.51|           63.7|    122.92|          0.0|Cash on Delivery|    5796 Main St|
|       3|       1381|2023-03-22|   2023-03-25| Returned|1191.88|          40.14|     95.35|          0.0|Cash on Delivery|    3354 Main St|
+--------+-----------+----------+-------------+---------+-------+---------------+----------+-------------+----------------+----------------+
only showing 

In [11]:
# --- CAST TYPE ---
orders_df.withColumn("total_amount_str",col("total_amount").cast("string")).printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- shipping_cost: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_address: string (nullable = true)
 |-- total_amount_str: string (nullable = true)



In [12]:
# --- DISTINCT ---

orders_df.select("status").distinct().count()

5

In [17]:
# --- DROP DUPLICATE ---
print(f"Row before dedup: {orders_df.count()}")

print(f"Rows after dedup (by customer_id) : {orders_df.dropDuplicates(['customer_id']).count()}")

Row before dedup: 50000
Rows after dedup (by customer_id) : 5000


In [15]:
# 2.1 All DataFrame Actions

# --- count() --- Returns number of rows
print(f"Total orders: {orders_df.count()}")

# --- show(n) --- Displays first n rows
print("\nFirst 5 orders:")
orders_df.show(5)

# --- first() --- Returns first Row
print(f"\nFirst row: {orders_df.first()}")

# --- head(n) --- Returns first n Rows
print(f"\nHead 3: {orders_df.head(3)}")

# --- take(n) --- Returns first n Rows (same as head)
print(f"\nTake 3: {orders_df.take(3)}")


Total orders: 50000

First 5 orders:
+--------+-----------+----------+-------------+---------+------------+---------------+----------+-------------+----------------+----------------+
|order_id|customer_id|order_date|delivery_date|   status|total_amount|discount_amount|tax_amount|shipping_cost|  payment_method|shipping_address|
+--------+-----------+----------+-------------+---------+------------+---------------+----------+-------------+----------------+----------------+
|       1|       4722|2023-08-08|   2023-08-14|  Shipped|      1784.2|         163.19|    142.74|         5.99|   Bank Transfer|    5823 Main St|
|       2|       3172|2022-09-19|   2022-09-24|Delivered|     1536.51|           63.7|    122.92|          0.0|Cash on Delivery|    5796 Main St|
|       3|       1381|2023-03-22|   2023-03-25| Returned|     1191.88|          40.14|     95.35|          0.0|Cash on Delivery|    3354 Main St|
|       4|       2332|2022-10-05|   2022-10-10|Delivered|     1157.78|          63.95| 

In [16]:

# --- collect() --- Returns all rows (DANGEROUS for large datasets!)
# orders_df.collect()  # DON'T do this on large data!

# --- toPandas() --- Convert to Pandas DataFrame
pandas_df = orders_df.limit(100).toPandas()  # Safe with limit
print(f"Pandas DataFrame shape: {pandas_df.shape}")

# --- describe() --- Summary statistics
print("\nStatistics:")
orders_df.select("total_amount").describe().show()

# --- summary() --- Detailed statistics
print("\nDetailed Statistics:")
orders_df.select("total_amount").summary("count", "mean", "stddev", "min", "25%", "75%", "max").show()

Pandas DataFrame shape: (100, 11)

Statistics:
+-------+------------------+
|summary|      total_amount|
+-------+------------------+
|  count|             50000|
|   mean|1253.8203528000008|
| stddev| 717.0386331683346|
|    min|             10.02|
|    max|           2499.99|
+-------+------------------+


Detailed Statistics:
+-------+------------------+
|summary|      total_amount|
+-------+------------------+
|  count|             50000|
|   mean|1253.8203528000008|
| stddev| 717.0386331683346|
|    min|             10.02|
|    25%|            632.87|
|    75%|           1871.94|
|    max|           2499.99|
+-------+------------------+



In [ ]:
# 2.3 Writing (Action)

orders_df.write.mode("overwrite").parquet('data/processed/orders_patiotioned')
print("Write completed - this triggered computation")

Write completed - this triggered computation


In [20]:
# 3.1 Viewing the Execution Plan

# Simple plan query
simple_query = (
    orders_df.filter(col("status")=="Delivered")
    .groupBy("customer_id").agg(
        sum("total_amount").alias("total_spent")
    ).orderBy(desc("total_spent"))
)

print("===Logical Plan===")
simple_query.explain(True)

===Logical Plan===
== Parsed Logical Plan ==
'Sort ['total_spent DESC NULLS LAST], true
+- Aggregate [customer_id#18], [customer_id#18, sum(total_amount#22) AS total_spent#1065]
   +- Filter (status#21 = Delivered)
      +- Relation [order_id#17,customer_id#18,order_date#19,delivery_date#20,status#21,total_amount#22,discount_amount#23,tax_amount#24,shipping_cost#25,payment_method#26,shipping_address#27] csv

== Analyzed Logical Plan ==
customer_id: int, total_spent: double
Sort [total_spent#1065 DESC NULLS LAST], true
+- Aggregate [customer_id#18], [customer_id#18, sum(total_amount#22) AS total_spent#1065]
   +- Filter (status#21 = Delivered)
      +- Relation [order_id#17,customer_id#18,order_date#19,delivery_date#20,status#21,total_amount#22,discount_amount#23,tax_amount#24,shipping_cost#25,payment_method#26,shipping_address#27] csv

== Optimized Logical Plan ==
Sort [total_spent#1065 DESC NULLS LAST], true
+- Aggregate [customer_id#18], [customer_id#18, sum(total_amount#22) AS total

In [22]:
# 3.2 Understanding Stages and Tasks


# Complex query with multiple shuffles

complex_query = (
    orders_df
    .join(customers_df,"customer_id")
    .filter(col("status")=="Delivered")
    .groupBy("customer_segment","city")
    .agg(
        count("*").alias("order_count"),
        sum("total_amount").alias("total_revenue")
    ).orderBy(desc("total_revenue"))
)

print("==Complex Query Plan ==")
complex_query.explain()

==Complex Query Plan ==
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_revenue#1082 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_revenue#1082 DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=863]
      +- HashAggregate(keys=[customer_segment#53, city#50], functions=[count(1), sum(total_amount#22)])
         +- Exchange hashpartitioning(customer_segment#53, city#50, 8), ENSURE_REQUIREMENTS, [plan_id=860]
            +- HashAggregate(keys=[customer_segment#53, city#50], functions=[partial_count(1), partial_sum(total_amount#22)])
               +- Project [total_amount#22, city#50, customer_segment#53]
                  +- BroadcastHashJoin [customer_id#18], [customer_id#45], Inner, BuildRight, false
                     :- Project [customer_id#18, total_amount#22]
                     :  +- Filter ((isnotnull(status#21) AND (status#21 = Delivered)) AND isnotnull(customer_id#18))
                     :     +- FileScan csv [customer_id#18,statu

In [26]:
# 4.1 Filter Early Example

import time

start =  time.time()

bad_result = (
    orders_df.groupBy("customer_id").agg(
        sum("total_amount").alias("total")
    ).filter(col("total")>1000) # Filter after group by
)

bad_result.show(5)
print(f"Bad approach: {time.time() - start:2f}s")

# Good:Filter Before expensive operation

start = time.time()
good_result = (
    orders_df.filter(col("status")=="Delivered")
    .groupBy("customer_id").agg(
        sum("total_amount").alias("total")
    ).filter(col("total")>1000) # Filter before groupby
)
good_result.show(5)
print(f"Good approach: {time.time() - start:2f}s")

+-----------+------------------+
|customer_id|             total|
+-----------+------------------+
|       2868| 7700.089999999999|
|       3344|          10275.36|
|       2908|16589.899999999998|
|       1088| 9895.869999999999|
|        787|           6467.13|
+-----------+------------------+
only showing top 5 rows
Bad approach: 0.169713s
+-----------+-----------------+
|customer_id|            total|
+-----------+-----------------+
|       3344|8553.380000000001|
|       2908|          9363.84|
|       1088|4647.200000000001|
|        787|          5350.24|
|       4162|         10526.14|
+-----------+-----------------+
only showing top 5 rows
Good approach: 0.175326s


In [27]:
spark.stop()
print("SparkSession Stopped")

SparkSession Stopped
